# 4.1 月度特征

给每个 SKU 的每个月算几个"预测那一刻就能知道"的数：前 1～3 月的需求量、近 3 月和近 6 月均值、
同类目季节系数。关键是绝不能用到当月或以后的数据——那叫数据泄漏，评估会好得不真实，一上线就露馅。

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

import dsflow

ROOT = Path.cwd()
while not (ROOT / "dsflow.yaml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from demo_lib import add_history, add_season, out, season_table

OUT = out("4.1")
PANEL = out("3.1") / "panel.parquet"
run = dsflow.start_run(
    "4.1", project=ROOT,
    hypothesis="只用预测月之前的信息构造特征：前 1～3 月需求量、近 3 / 6 月均值、同类目季节系数（只用训练期估计）")

run.log_input(PANEL, name="sku_month_panel")
panel = pd.read_parquet(PANEL)
table = season_table(panel[panel["划分"] == "训练"])
print(f"季节系数只用训练期估计，覆盖 {len(table)} 个类目")
print({c: dict(list(m.items())[:3]) for c, m in list(table.items())[:1]})


季节系数只用训练期估计，覆盖 3 个类目
{'MRO工业品': {'1': 0.9643, '2': 0.9746, '3': 0.9331}}


In [2]:
feat = add_season(add_history(panel), table)
before = len(feat)
feat = feat.dropna(subset=["mean6"]).reset_index(drop=True)
print(f"特征表 {before:,} 行 → {len(feat):,} 行（删掉历史不足 6 个月的 {before - len(feat):,} 行）")
feat.head(3)


特征表 72,000 行 → 54,000 行（删掉历史不足 6 个月的 18,000 行）


,SKU,类目,月份,需求量,划分,lag1,lag2,lag3,mean3,mean6,季节系数,season
0,SKU00001,办公通用物资,2025-01,2,训练,28.0,2.0,5.0,11.666667,14.666667,1.0290,12.005000
1,SKU00001,办公通用物资,2025-02,4,训练,2.0,28.0,2.0,10.666667,15.000000,0.8907,9.500800
2,SKU00001,办公通用物资,2025-03,39,训练,4.0,2.0,28.0,11.333333,10.666667,1.1821,13.397133


In [3]:
(OUT / "season_table.json").write_text(json.dumps(table, ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")
run.log_output(feat, name="features", path=OUT / "features.parquet", stage="features",
               description="SKU×月特征，一行 = 一个 SKU 的一个月；特征只用该月之前的数据")
run.log_metrics({"特征行": len(feat), "删除历史不足6个月的行": before - len(feat)})
run.log_artifact(OUT / "season_table.json", purpose="同类目季节系数（只用训练期估计）", kind="table")
counts = feat["划分"].value_counts().to_dict()
print(counts)


{'训练': 36000, '验证': 9000, '最终评估': 9000}


In [4]:
conclusion = (
    f"特征 {len(feat):,} 行（删除历史不足 6 个月的 {before - len(feat):,} 行）；"
    + "、".join(f"{k} {v:,} 行" for k, v in sorted(counts.items()))
)
run.set_conclusion(conclusion, validity="有效")
run.end()
print(conclusion)


特征 54,000 行（删除历史不足 6 个月的 18,000 行）；最终评估 9,000 行、训练 36,000 行、验证 9,000 行
